In [ ]:
import pandas as pd
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define paths
base_path = '/content/drive/My Drive/MIMIC/'
heart_path = base_path + 'HEART/heart_diagnoses.csv'
patients_path = base_path + 'HOSP/patients.csv.gz'
admissions_path = base_path + 'HOSP/admissions.csv.gz'

# 3. Load the datasets
df_heart = pd.read_csv(heart_path)
df_patients = pd.read_csv(patients_path, usecols=['subject_id', 'dod'])
df_admissions = pd.read_csv(admissions_path, usecols=['hadm_id', 'hospital_expire_flag'])

# 4. Merge the data
merged_data = pd.merge(df_heart, df_patients, on='subject_id', how='left')
final_df = pd.merge(merged_data, df_admissions, on='hadm_id', how='left')

# 5. Create survival logic
final_df['is_deceased'] = final_df['dod'].notna() | (final_df['hospital_expire_flag'] == 1)

# 6. Check available columns to avoid the KeyError
print("Available columns in your merged data:", final_df.columns.tolist())

# Select only the columns that actually exist in your dataframe
# This list checks if the column exists before trying to print it
cols_to_show = ['subject_id', 'hadm_id', 'hospital_expire_flag', 'is_deceased']
if 'icd_code' in final_df.columns:
    cols_to_show.append('icd_code')
elif 'icd9_code' in final_df.columns: # Sometimes used in older formats
    cols_to_show.append('icd9_code')

print(f"\nTotal Cardiac Records: {len(final_df)}")
print(final_df[cols_to_show].head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Available columns in your merged data: ['note_id', 'subject_id', 'hadm_id', 'note_type', 'note_seq', 'charttime', 'storetime', 'HPI', 'physical_exam', 'chief_complaint', 'invasions', 'X-ray', 'CT', 'Ultrasound', 'CATH', 'ECG', 'MRI', 'reports', 'dod', 'hospital_expire_flag', 'is_deceased']

Total Cardiac Records: 4761
   subject_id   hadm_id  hospital_expire_flag  is_deceased
0    10000980  29654838                     0         True
1    10000980  26913865                     0         True
2    10002013  24760295                     0        False
3    10002155  23822395                     0         True
4    10004457  28723315                     0        False


In [ ]:
final_df.shape

(4761, 21)

In [ ]:
# first

In [ ]:
import pandas as pd
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define paths based on your folder structure (image_d9f262.png)
base_path = '/content/drive/My Drive/MIMIC/'
diagnoses_path = base_path + 'HOSP/diagnoses_icd.csv.gz'
patients_path = base_path + 'HOSP/patients.csv.gz'
admissions_path = base_path + 'HOSP/admissions.csv.gz'

# 3. Load the full diagnosis table
print("Loading full diagnoses (this may take a moment)...")
df_all_diag = pd.read_csv(diagnoses_path)

# 4. Filter for Cardiac Diseases (Maximum Extraction)
# ICD-10: Starts with 'I' (e.g., I10, I21, I50)
# ICD-9: Range 390-459
df_cardiac = df_all_diag[
    (df_all_diag['icd_code'].str.startswith('I', na=False)) |
    (df_all_diag['icd_code'].str.startswith(('39', '40', '41', '42', '43', '44', '45'), na=False))
].copy()

# 5. Load metadata
df_patients = pd.read_csv(patients_path, usecols=['subject_id', 'dod'])
df_admissions = pd.read_csv(admissions_path, usecols=['hadm_id', 'hospital_expire_flag'])

# 6. Perform the Joins
merged = pd.merge(df_cardiac, df_patients, on='subject_id', how='left')
final_df = pd.merge(merged, df_admissions, on='hadm_id', how='left')

# 7. Final Survival Logic
final_df['is_deceased'] = final_df['dod'].notna() | (final_df['hospital_expire_flag'] == 1)

print(f"\n--- Extraction Complete ---")
print(f"Total Cardiac Rows Extracted: {len(final_df)}")
print(final_df[['subject_id', 'hadm_id', 'icd_code', 'hospital_expire_flag', 'is_deceased']].head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading full diagnoses (this may take a moment)...

--- Extraction Complete ---
Total Cardiac Rows Extracted: 968975
   subject_id   hadm_id icd_code  hospital_expire_flag  is_deceased
0    10000032  29079034    45829                     0         True
1    10000117  22927623     I341                     0        False
2    10000117  27988844     I341                     0        False
3    10000635  20642640      I10                     0        False
4    10000635  20642640     I471                     0        False


In [ ]:
final_df.shape

(968975, 8)

In [ ]:
import pandas as pd
from google.colab import drive

# 1. Setup paths
base_path = '/content/drive/My Drive/MIMIC/HEART/'

# 2. Load all heart-related datasets
df_diag = pd.read_csv(base_path + 'heart_diagnoses_all_true.csv')
df_labs = pd.read_csv(base_path + 'heart_labevents_examination_group.csv')
df_proc = pd.read_csv(base_path + 'heart_procedures.csv')
df_notes = pd.read_csv(base_path + 'heart_diagnoses.csv')

# 3. FIX: Map subject_id to the lab data
# Create a mapping table from the diagnosis file which has both IDs
id_mapping = df_diag[['subject_id', 'hadm_id']].drop_duplicates()
df_labs = df_labs.merge(id_mapping, on='hadm_id', how='left')

# 4. Aggregate Lab Events (Mean values for numeric features)
# Now that subject_id is present, we can group and unstack labels like 'Troponin T'
labs_agg = df_labs.groupby(['subject_id', 'label'])['valuenum'].mean().unstack().reset_index()

# 5. Aggregate Diagnoses (Join unique heart conditions into one string)
diag_agg = df_diag.groupby('subject_id')['long_title'].apply(lambda x: ' | '.join(set(x))).reset_index()

# 6. Aggregate Procedures (Count total heart procedures per patient)
proc_agg = df_proc.groupby('subject_id')['long_title'].count().reset_index()
proc_agg.columns = ['subject_id', 'procedure_count']

# 7. Extract Clinical Notes (Take the most recent report text)
notes_agg = df_notes.sort_values('charttime').groupby('subject_id')['reports'].last().reset_index()

# 8. MASTER MERGE - Creating the Unique Patient Dataset
# Outer join ensures no patient is dropped even if they are missing certain features
final_dataset = diag_agg.merge(labs_agg, on='subject_id', how='outer') \
                        .merge(proc_agg, on='subject_id', how='outer') \
                        .merge(notes_agg, on='subject_id', how='outer')

# 9. Add Survival Metadata from HOSP folder
# Including anchor_age and gender as important features
df_patients = pd.read_csv('/content/drive/My Drive/MIMIC/HOSP/patients.csv.gz', usecols=['subject_id', 'gender', 'anchor_age', 'dod'])
final_dataset = final_dataset.merge(df_patients, on='subject_id', how='left')

# 10. Define Target Label
final_dataset['is_deceased'] = final_dataset['dod'].notna().astype(int)

# Display results
print(f"Final Unique Patients: {final_dataset['subject_id'].nunique()}")
print(f"Total Features Extracted: {len(final_dataset.columns)}")
print("\nFirst 5 rows of the unique patient dataset:")
print(final_dataset[['subject_id', 'is_deceased', 'procedure_count']].head())

# Save for your project
final_dataset.to_csv(base_path + 'heart_patient_master_unique.csv', index=False)

Final Unique Patients: 4289
Total Features Extracted: 510

First 5 rows of the unique patient dataset:
   subject_id  is_deceased  procedure_count
0    10000980            1              7.0
1    10002013            0              2.0
2    10002155            1              8.0
3    10004457            0              NaN
4    10007058            0              2.0


In [ ]:
final_dataset.shape

(4289, 510)

In [ ]:
# 2nd

In [ ]:
import pandas as pd
from google.colab import drive

# 1. Setup paths
base_path = '/content/drive/My Drive/MIMIC/HEART/'

# 2. Load all heart-related datasets
df_diag = pd.read_csv(base_path + 'heart_diagnoses_all_true.csv')
df_labs = pd.read_csv(base_path + 'heart_labevents_examination_group.csv')
df_proc = pd.read_csv(base_path + 'heart_procedures.csv')
df_notes = pd.read_csv(base_path + 'heart_diagnoses.csv')

# 3. Create a Master Visit List (HADM_ID level)
# This captures every unique heart-related admission across all files
all_visits = pd.concat([
    df_diag[['subject_id', 'hadm_id']],
    df_proc[['subject_id', 'hadm_id']],
    df_notes[['subject_id', 'hadm_id']]
]).drop_duplicates().dropna(subset=['hadm_id'])

# 4. Feature Extraction at the Admission Level
# Diagnoses: All codes assigned during THIS specific visit
diag_agg = df_diag.groupby(['subject_id', 'hadm_id'])['long_title'].apply(lambda x: ' | '.join(set(x.astype(str)))).reset_index()

# Labs: Mean values for markers recorded during THIS specific visit
labs_agg = df_labs.groupby(['hadm_id', 'label'])['valuenum'].mean().unstack().reset_index()

# Procedures: Count of heart procedures performed during THIS specific visit
proc_agg = df_proc.groupby(['subject_id', 'hadm_id'])['long_title'].count().reset_index()
proc_agg.columns = ['subject_id', 'hadm_id', 'visit_procedure_count']

# Notes: The report associated with THIS specific admission
notes_agg = df_notes.groupby(['subject_id', 'hadm_id'])['reports'].first().reset_index()

# 5. The Master Join (Joining everything on hadm_id)
final_dataset = all_visits.merge(diag_agg, on=['subject_id', 'hadm_id'], how='left') \
                          .merge(labs_agg, on='hadm_id', how='left') \
                          .merge(proc_agg, on=['subject_id', 'hadm_id'], how='left') \
                          .merge(notes_agg, on=['subject_id', 'hadm_id'], how='left')

# 6. Add Survival Metadata from HOSP folder
# hospital_expire_flag is the key here: did they die during THIS admission?
df_adm_meta = pd.read_csv('/content/drive/My Drive/MIMIC/HOSP/admissions.csv.gz',
                          usecols=['hadm_id', 'hospital_expire_flag', 'admission_type', 'insurance'])
df_patients = pd.read_csv('/content/drive/My Drive/MIMIC/HOSP/patients.csv.gz',
                          usecols=['subject_id', 'gender', 'anchor_age'])

final_dataset = final_dataset.merge(df_adm_meta, on='hadm_id', how='left')
final_dataset = final_dataset.merge(df_patients, on='subject_id', how='left')

# 7. Final Output
print(f"Total Heart-Related Admissions (Rows): {len(final_dataset)}")
print(f"Total Unique Features (Columns): {len(final_dataset.columns)}")

# Check for the mortality outcome
print("\nMortality Distribution (hospital_expire_flag):")
print(final_dataset['hospital_expire_flag'].value_counts())

final_dataset.to_csv(base_path + 'heart_admission_level_maximum.csv', index=False)

Total Heart-Related Admissions (Rows): 4761
Total Unique Features (Columns): 512

Mortality Distribution (hospital_expire_flag):
hospital_expire_flag
0    4659
1     102
Name: count, dtype: int64


In [ ]:
import pandas as pd
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')
base_path = '/content/drive/My Drive/MIMIC/'
heart_ext_path = base_path + 'HEART/'

# 2. Optimized Metadata Loading
# Loading only essential columns and setting data types to save RAM
print("Loading patient and admission metadata...")
df_patients = pd.read_csv(base_path + 'HOSP/patients.csv.gz',
                          usecols=['subject_id', 'gender', 'anchor_age', 'dod'],
                          dtype={'subject_id': 'int32'})

df_admissions = pd.read_csv(base_path + 'HOSP/admissions.csv.gz',
                            usecols=['hadm_id', 'hospital_expire_flag', 'admission_type'],
                            dtype={'hadm_id': 'int32', 'hospital_expire_flag': 'float32'})

# 3. Load and Aggregate Specialized Heart Features
print("Processing specialized heart features...")
df_labs = pd.read_csv(heart_ext_path + 'heart_labevents_examination_group.csv')
labs_agg = df_labs.groupby(['hadm_id', 'label'])['valuenum'].mean().unstack().reset_index()

df_proc = pd.read_csv(heart_ext_path + 'heart_procedures.csv')
proc_agg = df_proc.groupby(['hadm_id'])['long_title'].count().reset_index()
proc_agg.columns = ['hadm_id', 'visit_procedure_count']

# 4. Process the 9 Lakh+ Records in Chunks to prevent crashing
print("Processing large diagnosis file in chunks...")
diag_chunks = pd.read_csv(base_path + 'HOSP/diagnoses_icd.csv.gz', chunksize=100000)

final_chunks = []

for chunk in diag_chunks:
    # Filter for Heart/Circulatory only (ICD-10 'I' or ICD-9 390-459)
    cardiac_chunk = chunk[
        (chunk['icd_code'].str.startswith('I', na=False)) |
        (chunk['icd_code'].str.startswith(('39', '40', '41', '42', '43', '44', '45'), na=False))
    ].copy()

    # Merge with features
    temp = cardiac_chunk.merge(labs_agg, on='hadm_id', how='left')
    temp = temp.merge(proc_agg, on='hadm_id', how='left')
    temp = temp.merge(df_patients, on='subject_id', how='left')
    temp = temp.merge(df_admissions, on='hadm_id', how='left')

    final_chunks.append(temp)

# 5. Combine and finalize
final_df = pd.concat(final_chunks, ignore_index=True)

# Survival Logic
final_df['is_deceased'] = (final_df['dod'].notna()) | (final_df['hospital_expire_flag'] == 1)

print(f"\nSuccess! Total Rows: {len(final_df)}")
final_df.to_csv(heart_ext_path + 'massive_heart_dataset_optimized.csv', index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading patient and admission metadata...
Processing specialized heart features...
Processing large diagnosis file in chunks...

Success! Total Rows: 968975


In [ ]:
final_df.shape

(968975, 514)

In [ ]:
final_df.count()

,0
subject_id,968975
hadm_id,968975
seq_num,968975
icd_code,968975
icd_version,968975
...,...
anchor_age,968975
dod,399888
admission_type,968975
hospital_expire_flag,968975


In [ ]:
# This will print every single column name in order
print(final_df.columns.tolist())

['subject_id', 'hadm_id', 'seq_num', 'icd_code', 'icd_version', '% Hemoglobin A1c', '24 hr Calcium', '24 hr Creatinine', '24 hr Protein', '25-OH Vitamin D', 'ADP', 'ANCA Titer', 'ARCH-1', 'Absolute Basophil Count', 'Absolute CD3 Count', 'Absolute CD4 Count', 'Absolute CD8 Count', 'Absolute Eosinophil Count', 'Absolute Lymphocyte Count', 'Absolute Monocyte Count', 'Absolute Neutrophil Count', 'Acanthocytes', 'Acetaminophen', 'Acetone', 'Alanine Aminotransferase (ALT)', 'Albumin', 'Albumin, Ascites', 'Albumin, Body Fluid', 'Albumin, Pleural', 'Albumin, Urine', 'Albumin/Creatinine, Urine', 'Alkaline Phosphatase', 'Alpha-Fetoprotein', 'Alveolar-arterial Gradient', 'Ammonia', 'Ammonium Biurate', 'Amorphous Crystals', 'Amphetamine Screen, Urine', 'Amylase', 'Amylase, Ascites', 'Amylase, Body Fluid', 'Amylase, Pleural', 'Anat Path Hold', 'Anion Gap', 'Anisocytosis', 'Anti-Mitochondrial Antibody', 'Anti-Neutrophil Cytoplasmic Antibody', 'Anti-Nuclear Antibody', 'Anti-Nuclear Antibody, Titer', 

In [ ]:
print([col for col in final_df.columns if 'id' in col.lower()])

['subject_id', 'hadm_id', 'Albumin, Body Fluid', 'Amylase, Body Fluid', 'Arachadonic Acid', 'Bicarbonate, Other Fluid', 'Calcium, Body Fluid', 'Chloride', 'Chloride, Body Fluid', 'Chloride, Urine', 'Chloride, Whole Blood', 'Creatinine, Body Fluid', 'Glucose, Body Fluid', 'Hematocrit, Joint Fluid', 'Hematocrit, Other Fluid', 'Hemosiderin', 'LD, Body Fluid', 'Miscellaneous, Body Fluid', 'Parathyroid Hormone', 'Phosphate, Body Fluid', 'Potassium, Body Fluid', 'RBC, Joint Fluid', 'RBC, Other Fluid', 'Rheumatoid Factor', 'Sodium, Body Fluid', 'Thyroid Peroxidase Antibodies', 'Thyroid Stimulating Hormone', 'Tidal Volume', 'Total Protein, Body Fluid', 'Tricyclic Antidepressant Screen', 'Triglycerides', 'Triglycerides, Ascites', 'Triglycerides, Pleural', 'Urea Nitrogen, Body Fluid', 'Uric Acid', 'Uric Acid Crystals', 'Uric Acid, Urine', 'Valproic Acid', 'Voided Specimen']


In [ ]:
import pandas as pd
from google.colab import drive

# 1. Mount and Setup
drive.mount('/content/drive')
base_path = '/content/drive/My Drive/MIMIC/HEART/'

# 2. Load your massive dataset (the one with 514 columns)
df = pd.read_csv(base_path + 'massive_heart_dataset_optimized.csv')

# 3. Define the Mapping Groups
feature_groups = {
    'Cardiac_Markers': ['Troponin T', 'CK-MB Index', 'Creatine Kinase (CK)', 'proBNP', 'NTproBNP'],
    'Blood_Coagulation': ['INR(PT)', 'PT', 'PTT', 'Fibrinogen', 'Platelet Count', 'Thrombin'],
    'Electrolytes_Metabolic': ['Sodium', 'Potassium', 'Chloride', 'Bicarbonate', 'Calcium', 'Glucose', 'eAG'],
    'Immune_Inflammation': [col for col in df.columns if any(x in col for x in ['WBC', 'CD3', 'CD4', 'CD8', 'Immuno', 'Absolute'])],
    'Renal_Function': ['Creatinine', 'Urea Nitrogen', 'Albumin/Creatinine, Urine'],
    'Respiratory_BloodGas': ['pO2', 'pCO2', 'pH', 'PEEP', 'Ventilation Rate', 'O2 Flow'],
    'Viral_Infectious_Screen': [col for col in df.columns if any(x in col for x in ['HIV', 'Hepatitis', 'PCR', 'Virus'])]
}

# 4. Concatenate: Calculate the mean of existing values for each group
print("Concatenating features into Super-Categories...")
for group_name, cols in feature_groups.items():
    # Only use columns that actually exist in the dataframe
    existing_cols = [c for c in cols if c in df.columns]
    if existing_cols:
        df[group_name] = df[existing_cols].mean(axis=1)

# 5. Keep only the requested features + the new Super-Categories
keep_columns = [
    'subject_id', 'hadm_id', 'icd_code', 'icd_version', 'long_title', 'reports',
    'gender', 'anchor_age', 'dod', 'admission_type', 'hospital_expire_flag', 'is_deceased',
    'visit_procedure_count'
] + list(feature_groups.keys())

# Ensure we don't try to keep columns that don't exist
final_cols = [c for c in keep_columns if c in df.columns]
df_final = df[final_cols].copy()

# 6. Final Results
print(f"Extraction Complete. Reduced from 514 columns to {len(df_final.columns)} columns.")
df_final.to_csv(base_path + 'heart_concatenated_dataset.csv', index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from google.colab import drive

# 1. Setup
drive.mount('/content/drive')
base_path = '/content/drive/My Drive/MIMIC/HEART/'
file_path = base_path + 'massive_heart_dataset_optimized.csv'

# 2. Define the exact columns we need to extract
# Core columns you want to keep untouched
core_cols = [
    'subject_id', 'hadm_id', 'icd_code', 'icd_version', 'long_title', 'reports',
    'gender', 'anchor_age', 'dod', 'admission_type', 'hospital_expire_flag',
    'is_deceased', 'visit_procedure_count'
]

# Lab marker groups
cardiac_markers = ['Troponin T', 'CK-MB Index', 'Creatine Kinase (CK)', 'proBNP', 'NTproBNP']
coagulation = ['INR(PT)', 'PT', 'PTT', 'Fibrinogen', 'Platelet Count', 'Thrombin']
metabolic = ['Sodium', 'Potassium', 'Chloride', 'Bicarbonate', 'Calcium', 'Glucose', 'eAG']
renal = ['Creatinine', 'Urea Nitrogen', 'Albumin/Creatinine, Urine']
respiratory = ['pO2', 'pCO2', 'pH', 'PEEP', 'Ventilation Rate', 'O2 Flow']

# Combine all desired columns into one list
all_needed_cols = core_cols + cardiac_markers + coagulation + metabolic + renal + respiratory

# 3. Load ONLY the needed columns to save RAM
print("Loading selective columns from large dataset...")
# We use usecols to ignore the 400+ columns we don't want
df = pd.read_csv(file_path, usecols=lambda x: x in all_needed_cols)

# 4. Perform Concatenation (Average the groups)
print("Creating Super-Categories...")
df['Cardiac_Markers'] = df[[c for c in cardiac_markers if c in df.columns]].mean(axis=1)
df['Blood_Coagulation'] = df[[c for c in coagulation if c in df.columns]].mean(axis=1)
df['Electrolytes_Metabolic'] = df[[c for c in metabolic if c in df.columns]].mean(axis=1)
df['Renal_Function'] = df[[c for c in renal if c in df.columns]].mean(axis=1)
df['Respiratory_BloodGas'] = df[[c for c in respiratory if c in df.columns]].mean(axis=1)

# 5. Filter for Final Column Set
# Drop the individual lab markers now that we have the averages
final_cols = core_cols + ['Cardiac_Markers', 'Blood_Coagulation', 'Electrolytes_Metabolic', 'Renal_Function', 'Respiratory_BloodGas']
df_final = df[[c for c in final_cols if c in df.columns]].copy()
df_final = df_final.drop(columns=['dod'], errors='ignore')
# 6. Save and Clean Up
print(f"Extraction Complete! Final shape: {df_final.shape}")
df_final.to_csv(base_path + 'heart_concatenated_dataset.csv', index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading selective columns from large dataset...
Creating Super-Categories...
Extraction Complete! Final shape: (968975, 15)


In [ ]:
df_final.isnull().sum()

,0
subject_id,0
hadm_id,0
icd_code,0
icd_version,0
gender,0
anchor_age,0
admission_type,0
hospital_expire_flag,0
is_deceased,0
visit_procedure_count,951220


In [ ]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
base_path = '/content/drive/My Drive/MIMIC/HEART/'
file_path = base_path + 'massive_heart_dataset_optimized.csv'

# 1. Selective Loading: Only load the columns we actually want to keep
# We are including the 'long_title' and 'reports' specifically now
important_cols = [
    'subject_id', 'hadm_id', 'icd_code', 'long_title', 'reports',
    'gender', 'anchor_age', 'hospital_expire_flag', 'is_deceased',
    'visit_procedure_count', 'Cardiac_Markers', 'Blood_Coagulation',
    'Electrolytes_Metabolic', 'Renal_Function', 'Respiratory_BloodGas'
]

print("Loading high-priority columns...")
# Use lambda to load only existing important columns
df = pd.read_csv(file_path, usecols=lambda x: x in important_cols)

# 2. Optimization: Drop rows that have NO heart-specific clinical data
# This removes the 900,000+ 'empty' rows and keeps only the high-quality ones
print("Dropping empty clinical rows to optimize memory...")
clinical_features = ['reports', 'visit_procedure_count', 'Cardiac_Markers', 'Blood_Coagulation']
df = df.dropna(subset=[c for c in clinical_features if c in df.columns], how='all')

# 3. Final Result
print(f"Final Optimized Shape: {df.shape}")
print(f"Total Rows with Real Heart Data: {len(df)}")

# Save to Excel or CSV
# If it's under 1 million rows (which it will be now), it will open easily
df.to_csv(base_path + 'final_heart_clinical_master.csv', index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading high-priority columns...
Dropping empty clinical rows to optimize memory...
Final Optimized Shape: (17755, 8)
Total Rows with Real Heart Data: 17755


In [ ]:
df.head()

,subject_id,hadm_id,icd_code,visit_procedure_count,gender,anchor_age,hospital_expire_flag,is_deceased
64,10000980,26913865,41071,7.0,F,73,0.0,True
65,10000980,26913865,42823,7.0,F,73,0.0,True
66,10000980,26913865,41412,7.0,F,73,0.0,True
67,10000980,26913865,4240,7.0,F,73,0.0,True
68,10000980,26913865,4280,7.0,F,73,0.0,True


In [ ]:
# input_path = '/content/drive/My Drive/MIMIC/HEART/massive_heart_dataset_optimized.csv'
# # Target path (where you want the Excel file to go)
# output_folder = '/content/drive/My Drive/MIMIC CARDIAC/'
# output_file = output_folder + 'final_heart_clinical_master.xlsx'

In [ ]:
import pandas as pd

base_path = '/content/drive/My Drive/MIMIC/HEART/'

# 1. Load your master file and the text source files
df_master = pd.read_csv(base_path + 'final_heart_clinical_master.csv')
df_notes = pd.read_csv(base_path + 'heart_diagnoses.csv', usecols=['hadm_id', 'reports'])
df_proc = pd.read_csv(base_path + 'heart_procedures.csv', usecols=['hadm_id', 'long_title'])

# 2. Drop duplicates before merging to keep row counts stable
df_notes = df_notes.drop_duplicates(subset=['hadm_id'])
df_proc = df_proc.groupby('hadm_id')['long_title'].apply(lambda x: ' | '.join(set(x))).reset_index()

# 3. Merge text columns into your master file
df_master = df_master.merge(df_notes, on='hadm_id', how='left')
df_master = df_master.merge(df_proc, on='hadm_id', how='left')

print(f"Text extraction complete. Shape: {df_master.shape}")

Text extraction complete. Shape: (17755, 10)


In [ ]:
df_master.head()

,subject_id,hadm_id,icd_code,visit_procedure_count,gender,anchor_age,hospital_expire_flag,is_deceased,reports,long_title
0,10000980,26913865,41071,7.0,F,73,0.0,True,Sinus bradycardia with sinus arrhythmia | Prol...,Coronary arteriography using two catheters | L...
1,10000980,26913865,42823,7.0,F,73,0.0,True,Sinus bradycardia with sinus arrhythmia | Prol...,Coronary arteriography using two catheters | L...
2,10000980,26913865,41412,7.0,F,73,0.0,True,Sinus bradycardia with sinus arrhythmia | Prol...,Coronary arteriography using two catheters | L...
3,10000980,26913865,4240,7.0,F,73,0.0,True,Sinus bradycardia with sinus arrhythmia | Prol...,Coronary arteriography using two catheters | L...
4,10000980,26913865,4280,7.0,F,73,0.0,True,Sinus bradycardia with sinus arrhythmia | Prol...,Coronary arteriography using two catheters | L...


In [ ]:
base_path = '/content/drive/My Drive/MIMIC/HEART/'
df_master.to_csv(base_path + 'final_heart_clinical_master222222.csv', index=False)

In [ ]:
# NLP

In [ ]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Load your dataset
base_path = '/content/drive/My Drive/MIMIC/HEART/'
df = pd.read_csv(base_path + 'final_heart_clinical_master222222.csv')
print(f"Original dataset shape: {df.shape}")

# 2. Preprocess the text columns (Fill empty/NaN spaces with empty strings)
df['reports'] = df['reports'].fillna('')
df['long_title'] = df['long_title'].fillna('')

# 3. NLP on 'reports' column (Extracting top 50 medical terms/phrases)
print("Performing NLP on clinical reports...")
tfidf_reports = TfidfVectorizer(
    max_features=50,
    stop_words='english',
    ngram_range=(1, 2) # Captures both single words ("bradycardia") and phrases ("sinus rhythm")
)
reports_tfidf_matrix = tfidf_reports.fit_transform(df['reports']).toarray()

# Convert the matrix into a clean DataFrame with words as column names
df_reports_nlp = pd.DataFrame(
    reports_tfidf_matrix,
    columns=[f"report_{word.replace(' ', '_')}" for word in tfidf_reports.get_feature_names_out()]
)

# 4. NLP on 'long_title' column (Extracting top 30 procedure/diagnosis terms)
print("Performing NLP on long titles...")
tfidf_titles = TfidfVectorizer(
    max_features=30,
    stop_words='english',
    ngram_range=(1, 2)
)
titles_tfidf_matrix = tfidf_titles.fit_transform(df['long_title']).toarray()

# Convert the matrix into a clean DataFrame with words as column names
df_titles_nlp = pd.DataFrame(
    titles_tfidf_matrix,
    columns=[f"title_{word.replace(' ', '_')}" for word in tfidf_titles.get_feature_names_out()]
)

# 5. Merge the new NLP features back with your original columns
# We reset indices to align them perfectly before concatenation
df_core = df.drop(columns=['reports', 'long_title']).reset_index(drop=True)
df_nlp_final = pd.concat([df_core, df_reports_nlp, df_titles_nlp], axis=1)

# 6. Save the new dataset
print(f"NLP complete! Expanded dataset shape: {df_nlp_final.shape}")
df_nlp_final.to_csv(base_path + 'heart_clinical_nlp_ready.csv', index=False)
print("Saved new dataset as 'heart_clinical_nlp_ready.csv'")

# Quick look at some of the medical terms extracted as features
print("\nSample of new medical text features created:")
print(list(df_nlp_final.columns[-10:]))

Original dataset shape: (17755, 10)
Performing NLP on clinical reports...
Performing NLP on long titles...
NLP complete! Expanded dataset shape: (17755, 88)
Saved new dataset as 'heart_clinical_nlp_ready.csv'

Sample of new medical text features created:
['title_percutaneous', 'title_percutaneous_approach', 'title_procedure', 'title_right', 'title_single', 'title_stent', 'title_transluminal', 'title_transluminal_coronary', 'title_using', 'title_using_catheters']


In [ ]:
df_nlp_final.head()

,subject_id,hadm_id,icd_code,visit_procedure_count,gender,anchor_age,hospital_expire_flag,is_deceased,report_1st,report_1st_degree,...,title_percutaneous,title_percutaneous_approach,title_procedure,title_right,title_single,title_stent,title_transluminal,title_transluminal_coronary,title_using,title_using_catheters
0,10000980,26913865,41071,7.0,F,73,0.0,True,0.0,0.0,...,0.139635,0.0,0.3679,0.0,0.0,0.382986,0.186404,0.186404,0.116653,0.145704
1,10000980,26913865,42823,7.0,F,73,0.0,True,0.0,0.0,...,0.139635,0.0,0.3679,0.0,0.0,0.382986,0.186404,0.186404,0.116653,0.145704
2,10000980,26913865,41412,7.0,F,73,0.0,True,0.0,0.0,...,0.139635,0.0,0.3679,0.0,0.0,0.382986,0.186404,0.186404,0.116653,0.145704
3,10000980,26913865,4240,7.0,F,73,0.0,True,0.0,0.0,...,0.139635,0.0,0.3679,0.0,0.0,0.382986,0.186404,0.186404,0.116653,0.145704
4,10000980,26913865,4280,7.0,F,73,0.0,True,0.0,0.0,...,0.139635,0.0,0.3679,0.0,0.0,0.382986,0.186404,0.186404,0.116653,0.145704


In [ ]:
import pandas as pd
from google.colab import drive

# 1. Setup paths
drive.mount('/content/drive')
base_path = '/content/drive/My Drive/MIMIC/HEART/'
icu_path = '/content/drive/My Drive/MIMIC/ICU/chartevents.csv.gz'

# 2. Load your NLP-ready dataset (17,000+ rows)
df_master = pd.read_csv(base_path + 'heart_clinical_nlp_ready.csv')
print(f"Master dataset loaded. Total rows: {len(df_master)}")

# 3. Define the vital sign Item IDs from MIMIC-IV
# 220045 = Heart Rate (bpm)
# 220179 = Non-Invasive Blood Pressure Systolic (mmHg)
# 220180 = Non-Invasive Blood Pressure Diastolic (mmHg)
# 220277 = O2 Saturation / SpO2 (%)
# 223761 = Temperature Fahrenheit (°F)
vitals_item_ids = [220045, 220179, 220180, 220277, 223761]

print("Scanning chartevents for matching patient vitals (processing in chunks)...")
chunk_container = []

# 4. Read chartevents in chunks to protect RAM
for chunk in pd.read_csv(icu_path, chunksize=500000, usecols=['hadm_id', 'itemid', 'valuenum']):
    # Only pull rows matching our 17k hadm_ids and our target vital signs
    filtered_chunk = chunk[(chunk['hadm_id'].isin(df_master['hadm_id'])) & (chunk['itemid'].isin(vitals_item_ids))]
    chunk_container.append(filtered_chunk)

# Combine the raw vital measurements
df_vitals_raw = pd.concat(chunk_container, ignore_index=True)

# 5. Pivot rows into clean columns (calculating the mean value for each admission)
print("Pivoting vital signs into columns...")
df_vitals_pivoted = df_vitals_raw.groupby(['hadm_id', 'itemid'])['valuenum'].mean().unstack().reset_index()

# Rename the columns so they make absolute sense
df_vitals_pivoted.columns = [
    'hadm_id',
    'vital_mean_heart_rate',
    'vital_mean_systolic_bp',
    'vital_mean_diastolic_bp',
    'vital_mean_spo2',
    'vital_mean_temp'
]

# 6. Merge vitals back to the master list using a LEFT JOIN to preserve all 17,000+ rows
final_dataset = df_master.merge(df_vitals_pivoted, on='hadm_id', how='left')

# 7. Save the completely finalized data structure
print(f"Vitals added successfully! Final dataset shape: {final_dataset.shape}")
final_dataset.to_csv(base_path + 'final_heart_clinical_master222222.csv', index=False)
print("Saved complete dataset as 'final_heart_clinical_master3.csv'")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Master dataset loaded. Total rows: 17755
Scanning chartevents for matching patient vitals (processing in chunks)...
Pivoting vital signs into columns...
Vitals added successfully! Final dataset shape: (17755, 93)
Saved complete dataset as 'final_heart_clinical_master3.csv'


In [ ]:
final_dataset.head()

,subject_id,hadm_id,icd_code,visit_procedure_count,gender,anchor_age,hospital_expire_flag,is_deceased,report_1st,report_1st_degree,...,title_stent,title_transluminal,title_transluminal_coronary,title_using,title_using_catheters,vital_mean_heart_rate,vital_mean_systolic_bp,vital_mean_diastolic_bp,vital_mean_spo2,vital_mean_temp
0,10000980,26913865,41071,7.0,F,73,0.0,True,0.0,0.0,...,0.382986,0.186404,0.186404,0.116653,0.145704,73.636364,142.454545,83.272727,98.909091,98.133333
1,10000980,26913865,42823,7.0,F,73,0.0,True,0.0,0.0,...,0.382986,0.186404,0.186404,0.116653,0.145704,73.636364,142.454545,83.272727,98.909091,98.133333
2,10000980,26913865,41412,7.0,F,73,0.0,True,0.0,0.0,...,0.382986,0.186404,0.186404,0.116653,0.145704,73.636364,142.454545,83.272727,98.909091,98.133333
3,10000980,26913865,4240,7.0,F,73,0.0,True,0.0,0.0,...,0.382986,0.186404,0.186404,0.116653,0.145704,73.636364,142.454545,83.272727,98.909091,98.133333
4,10000980,26913865,4280,7.0,F,73,0.0,True,0.0,0.0,...,0.382986,0.186404,0.186404,0.116653,0.145704,73.636364,142.454545,83.272727,98.909091,98.133333


In [ ]:
final_dataset.isnull().sum()

,0
subject_id,0
hadm_id,0
icd_code,0
visit_procedure_count,0
gender,0
...,...
vital_mean_heart_rate,10087
vital_mean_systolic_bp,10150
vital_mean_diastolic_bp,10150
vital_mean_spo2,10087


In [ ]:
import pandas as pd
import openpyxl
from openpyxl.utils import get_column_letter

# 1. Setup file paths (Reuses 'base_path' from your notebook)
csv_filename = 'final_heart_clinical_master222222.csv'
excel_filename = 'final_heart_clinical_master3.xlsx'
base_path = '/content/drive/My Drive/MIMIC/HEART/'
csv_path = base_path + csv_filename
excel_path = base_path + excel_filename

print(f"Reading CSV from: {csv_path}")

# 2. Load the clinical dataset
# low_memory=False handles mixed numeric/string data types in clinical rows smoothly
df = pd.read_csv(csv_path, low_memory=False)
print(f"Loaded master dataset. Shape: {df.shape}")

# 3. Export to Excel format
print("Writing data to Excel layout...")
df.to_excel(excel_path, index=False, engine='openpyxl')

# 4. Polish the spreadsheet (Auto-fit columns, freeze panes, add filters)
print("Applying professional formatting to the spreadsheet...")
wb = openpyxl.load_workbook(excel_path)
ws = wb.active
ws.title = "Clinical Master"

# Enable visual column filtering across all columns
ws.auto_filter.ref = ws.dimensions

# Freeze the top row so headers stay visible while scrolling down patient records
ws.freeze_panes = "A2"

# Style the header row (Dark slate accent with white text)
header_font = openpyxl.styles.Font(name="Arial", size=11, bold=True, color="FFFFFF")
header_fill = openpyxl.styles.PatternFill(start_color="2C3E50", end_color="2C3E50", fill_type="solid")

for cell in ws[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = openpyxl.styles.Alignment(horizontal="center", vertical="center")

# Automatically adjust column widths based on the longest value in each column
for col in ws.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = get_column_letter(col[0].column)
    # Extra padding so text or numbers aren't clipped by the filter dropdown arrows
    ws.column_dimensions[col_letter].width = max(max_len + 4, 12)

# Save the polished workbook
wb.save(excel_path)
print(f"🟢 Success! Formatted spreadsheet saved directly to Drive at:\n -> {excel_path}")

Reading CSV from: /content/drive/My Drive/MIMIC/HEART/final_heart_clinical_master222222.csv
Loaded master dataset. Shape: (17755, 93)
Writing data to Excel layout...
Applying professional formatting to the spreadsheet...
🟢 Success! Formatted spreadsheet saved directly to Drive at:
 -> /content/drive/My Drive/MIMIC/HEART/final_heart_clinical_master3.xlsx


In [ ]:
# https://gemini.google.com/share/f1aa21db5602
# step on how to split data to prevent data leakage